# Threshold: a guided tour

This notebook walks through the engine using the code in `src/threshold`, reading only the small
result tables in `artifacts/` (run `make all` first, or `make fake` for generated data). Every cell
is a few lines: the work lives in the package, not here.

In [ ]:
import pandas as pd

from threshold.config import ARTIFACTS, FAKE_ARTIFACTS, load_assumptions
from threshold.histogram import load_hist

folder = ARTIFACTS if (ARTIFACTS / "meta.json").exists() else FAKE_ARTIFACTS
hist = pd.read_parquet(folder / "score_hist.parquet")
val = load_hist(hist, "full", (5,))  # calibration month: where every cut-off is fitted
test = load_hist(hist, "full", (6, 7))  # the two unseen months
a = load_assumptions()
print(
    f"Reading {folder.name}/ · {test.applications:,.0f} test applications, {test.n_fraud.sum():,.0f} frauds"
)

## 1. Each action's value is a straight line in p

`envelope` walks the upper envelope of the four lines exactly. The band edges are where the lines cross.

In [ ]:
from threshold.economics import approve_decline_breakeven, envelope

econ = a.economics()
for band in envelope(econ):
    print(f"{band.name:8s} {band.start:7.2%} → {band.end:7.2%}")
print(f"approve = decline at V/(V+L) = {approve_decline_breakeven(econ):.2%}")

## 2. Against the simple alternatives

Every policy is fitted on month 5 and scored on months 6–7.

In [ ]:
from threshold.policy import compare, decision_bands, describe_bands

comp = compare(val, test, a)
print(describe_bands(decision_bands(comp.policy)))
comp.table[
    [
        "profit_per_1k",
        "genuine_onboarded_share",
        "fraud_stopped_share",
        "approve_rate",
        "verify_rate",
        "review_rate",
        "decline_rate",
    ]
].round(3)

## 3. What is one more analyst worth?

For each team size the policy is re-fitted with that many review slots. Each analyst works the
most valuable cases left, so each adds less than the last.

In [ ]:
from threshold.policy import recommended_team, team_curve

curve = team_curve(val, test, a)
print("recommended team:", recommended_team(curve))
curve[["analysts", "reviews_per_day", "slot_value", "marginal_value_per_day", "worth_hiring"]].head(
    12
).round(1)

## 4. What is each data source worth?

The model was retrained without each supplier's features; the drop in profit per application is
the most to pay per check.

In [ ]:
from threshold.vendors import vendor_values

vendor_values(hist, a)[
    ["source", "value_per_application", "fraud_stopped_with", "fraud_stopped_without"]
].round(3)

## 5. Who bears the extra checks?

In [ ]:
from threshold.fairness import fairness_table

f = fairness_table(hist, a, "full")
f[f["metric"] == "genuine_checked_rate"][
    ["policy_label", "under_50", "over_50", "ratio", "flag"]
].round(3)

## 6. Month by month

In [ ]:
from threshold.monitoring import monthly_monitor

summary, bands = monthly_monitor(hist, a)
summary[
    [
        "month",
        "role",
        "approve_rate",
        "fraud_stopped_share",
        "score_psi",
        "score_status",
        "calibration_gap",
        "calibration_status",
    ]
].round(3)

## 7. Your turn

Change any assumption and re-run the comparison. The interview demo: loss per fraud from £500 to £2,000.

In [ ]:
for loss in (500, 1000, 2000):
    c = compare(val, test, a.with_(L=loss))
    t = c.table.loc["threshold"]
    print(
        f"L = £{loss:>5,}: {describe_bands(decision_bands(c.policy))}  |  "
        f"approved {t['genuine_onboarded_share']:.1%}, profit per 1,000 £{t['profit_per_1k']:,.0f}"
    )